# Frameflow Tier 2 — GPU verification

Runs `verify_gpu.py`, the one thing that cannot be tested without CUDA.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. The free tier
is enough; the scene is 16 frames at 192x108.

No SSH is involved. Colab disallows SSH on free runtimes, but nothing here needs
a shell -- the toolkit runs as an ordinary script.

## 1. Check you actually got a GPU

In [ ]:
!nvidia-smi -L
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

## 2. Upload the toolkit

Zip your `frameflow` folder locally and upload it here. The cell copies
the `.py` files to a clean `/content/toolkit` before running anything -- see the
comment inside for why that is not optional on a non-English Windows.

In [ ]:
from google.colab import files
import zipfile, pathlib, io, shutil, unicodedata

up = files.upload()                       # pick frameflow.zip
name = next(iter(up))
with zipfile.ZipFile(io.BytesIO(up[name])) as z:
    z.extractall('/content/tk')

src = next((p.parent for p in pathlib.Path('/content/tk').rglob('verify_gpu.py')), None)
assert src, "verify_gpu.py not found in the zip"

# Re-home the .py files under a clean ASCII path before doing anything else.
# A zip made on a Hebrew (or any RTL) Windows carries invisible direction marks
# in its folder name -- measured: two U+200F marks prefixing the folder
# name, invisible in every file manager. Those
# survive into the path, and `!python foo.py` then fails with a baffling
# "can't open file", because the shell and the notebook disagree about what
# the directory is called. Copying out sidesteps it permanently.
root = pathlib.Path('/content/toolkit')
shutil.rmtree(root, ignore_errors=True)
root.mkdir(parents=True)
for f in src.iterdir():
    if f.is_file():
        shutil.copy2(f, root / f.name)

odd = [c for c in str(src) if unicodedata.category(c) == 'Cf']
print('extracted from :', repr(str(src)))
print('direction marks:', len(odd), '-- this is why we copy out' if odd else '(none)')
%cd {root}
print(len(list(root.glob('*.py'))), 'py files')
!ls

## 3. Install gsplat

**The install is instant. The 5-10 minute wait happens later, and it is not a
hang.** gsplat ships as a `py3-none-any` wheel with no compiled code in it, so
pip finishes in seconds -- then the CUDA kernels JIT-compile the first time you
actually rasterize. Measured on a T4: **804 seconds inside the first
`fit_splats` call**, with `nvidia-smi` showing 0% GPU and two `ptxas` processes
burning CPU. That is the compile, not a stall. It is cached for the life of the
runtime, so every later run is fast.

If pip itself errors, read the last few lines rather than re-running -- a
torch/CUDA version mismatch will not fix itself on retry.

In [ ]:
!pip -q install opencv-python-headless
!pip install gsplat

## 4. Verify

In [ ]:
!python tools/verify_gpu.py

### What you should see

Every check `ok`, ending with the line about known truth. The ones that matter:

- **centre stays aligned under widening** -- if this fails nothing else means
  anything, the fence would reject every frame
- **recovered wings match the truth render** -- above ~15 dB. Coverage alone
  cannot tell you the splats are in the *right place*; this can
- **held-out frames score above the gate's threshold** -- above 20 dB, which is
  what stops the gate turning every 3D shot OFF
- **primary region is the original frame, byte for byte**

A failure here is information, not a setback -- it means the reconstruction is
wrong in a specific, named way, which is the entire point of writing the check
before spending money on a bigger GPU.

### Keep the proof

`verify_gpu.py` writes `verify_gpu_result.json` next to itself: every check,
its number, and which GPU produced it. Download it and commit it beside the
roadmap — otherwise a week later nobody can tell whether Tier 2 was *written*
or *verified*, which are very different states.


In [ ]:
import json
r = json.load(open('verify_gpu_result.json'))
print(r['environment'].get('device'), '| gsplat', r['environment'].get('gsplat'))
print(r['passed'], 'passed,', r['failed'], 'failed  ->  ok =', r['ok'])
for c in r['checks']:
    print(('  ok  ' if c['passed'] else '  FAIL'), c['name'], '--', c['detail'])

from google.colab import files
files.download('verify_gpu_result.json')


## 5. Optional: your own footage

Only worth doing once section 4 is green. COLMAP is the slow part -- minutes to
hours depending on shot count -- and Colab will recycle the runtime out from
under a long run, so start with `--max-shots 12`.

**Two things this cell gets right that are easy to get wrong.**

`--also` is what makes `RETRIEVED` reachable at all. One clip is one setup per
scene, and a lone setup has no second viewpoint to recover a wing from. Pass the
other setups of the same location and they are declared one scene rather than
guessed into one.

**Your footage has to have baseline.** Two-second locked-off clips will not
register: measured, 48 frames across two setups gave 58 points, split into two
disconnected models, 48% registered -- refused. Want 20-30 seconds per setup
with the camera genuinely moving. A continuous handheld or drone move is ideal;
that is what SfM is built to solve.

In [ ]:
!apt-get -qq install -y colmap

# Upload your clips first (folder icon in the left sidebar), then:
#
# !python -m frameflow.render /content/clipA.mp4 -o /content/out #      --also /content/clipB.mp4 #      --maxw 480 --max-shots 12 --sfm /content/sfm --prefer-3d
#
# COLMAP's SIFT runs on CPU here -- its GPU path needs a GL context and Colab
# is headless, so the GPU path dies with a perfectly good T4 sitting idle.
# sfm.build_scene picks CPU automatically when $DISPLAY is unset; no flag
# needed. Watch for the registration line: below 80% the scene is REFUSED
# rather than rendered from untrustworthy poses.

## 6. Get the results back

Colab's filesystem is ephemeral. Download before the runtime recycles.

In [ ]:
!zip -qr /content/out.zip /content/out /content/sfm 2>/dev/null || true
from google.colab import files
files.download('/content/out.zip')